# Junk Food Binary Classification with YOLO 11

This notebook implements a **YOLO 11** model for binary image classification (`junk-food-ad` vs `non-junk-food-ad`) using the **leakage-aware binary dataset** (`binary-dataset.zip`, folders `ad` / `non_ad` + `groups.csv`) stored in Google Drive.

The train/val/test split is **group-aware**: all frames of the same ad creative or source video stay in the same split, so near-duplicate frames cannot leak from train into validation/test.

## Before you start

Make sure you have access to GPU. In case of any problems, navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, click `Save` and try again.

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print("HOME:", HOME)

In [ ]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets


## Install packages using pip

In [ ]:
!pip install torch==2.9.0 torchvision==0.24.0 scikit-learn==1.6.1 tqdm==4.67.1 ultralytics==8.3.241

## Load dataset from Google Drive

Mount your Google Drive and add a Colab secret named `BINARY_DATASET_ZIP_PATH` (Secrets panel - the key icon in the left sidebar) pointing to `binary-dataset.zip` in Drive (absolute, e.g. `/content/drive/MyDrive/datasets/binary-dataset.zip`, or relative to `/content/drive`, e.g. `MyDrive/datasets/binary-dataset.zip`).

The zip (produced by `unhealthy-dataset-generator`) contains `ad/` (junk food ads), `non_ad/` (everything else) and `groups.csv`, which assigns every frame to a **group**: frames of the same ad creative (`ad_XXXX`) or of the same source video (`non_ad_<video>`). `groups.csv` may also list frames that were capped/rebalanced out of the zip; those rows are ignored. The notebook checks that every image has exactly one group and that the CSV label matches its folder.

In [ ]:
import os
import shutil
import zipfile
import numpy as np
import pandas as pd
from google.colab import drive, userdata

drive.mount('/content/drive')

# Path to binary-dataset.zip in your Google Drive. Add BINARY_DATASET_ZIP_PATH in Colab's
# "Secrets" panel (key icon in the left sidebar), e.g.
# /content/drive/MyDrive/Colab Notebooks/Tesis/datasets/binary-dataset.zip
_zip_secret = userdata.get('BINARY_DATASET_ZIP_PATH').strip()
DATASET_ZIP_PATH = (_zip_secret if _zip_secret.startswith('/content/')
                    else os.path.join('/content/drive', _zip_secret.lstrip('/')))
if not os.path.isfile(DATASET_ZIP_PATH):
    raise FileNotFoundError(f"Dataset zip not found: {DATASET_ZIP_PATH}")

# Copy the zip to local Colab storage and extract it there (much faster than reading from Drive).
# Always start from a clean folder so a previous/partial extraction can't leave stale files behind.
LOCAL_ZIP_PATH = "/content/binary-dataset.zip"
EXTRACT_ROOT = "/content/binary_dataset"
shutil.copy(DATASET_ZIP_PATH, LOCAL_ZIP_PATH)
if os.path.exists(EXTRACT_ROOT):
    shutil.rmtree(EXTRACT_ROOT)
with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_ROOT)

# Locate the folder that contains ad/, non_ad/ and groups.csv
DATASET_ROOT = None
for current_dir, subdirs, file_names in os.walk(EXTRACT_ROOT):
    if "ad" in subdirs and "non_ad" in subdirs and "groups.csv" in file_names:
        DATASET_ROOT = current_dir
        break
if DATASET_ROOT is None:
    raise FileNotFoundError("Could not find 'ad/', 'non_ad/' and 'groups.csv' inside the extracted dataset.")
print("Dataset folder:", DATASET_ROOT)

# Map source folders to binary class names. The positive class maps to index 0.
CLASS_FOLDERS = {
    "ad": "junk-food-ad",          # positive class
    "non_ad": "non-junk-food-ad",  # negative class
}
POSITIVE_CLASS = "junk-food-ad"
NEGATIVE_CLASS = "non-junk-food-ad"
IMG_EXTS = (".jpg", ".jpeg", ".png")

# Images actually present in the zip
disk_rows = []
for folder, class_name in CLASS_FOLDERS.items():
    folder_path = os.path.join(DATASET_ROOT, folder)
    for file_name in sorted(os.listdir(folder_path)):
        if file_name.lower().endswith(IMG_EXTS):
            disk_rows.append({"image_name": file_name, "folder": folder, "class_name": class_name,
                              "path": os.path.join(folder_path, file_name)})
disk_df = pd.DataFrame(disk_rows)
duplicated = disk_df[disk_df["image_name"].duplicated(keep=False)]
if not duplicated.empty:
    raise ValueError(f"Image names repeated across folders: {duplicated['image_name'].unique()[:10].tolist()}")

groups_df = pd.read_csv(os.path.join(DATASET_ROOT, "groups.csv"), dtype=str, encoding="utf-8-sig")
missing_cols = {"image_name", "label", "group"} - set(groups_df.columns)
if missing_cols:
    raise ValueError(f"groups.csv is missing columns: {sorted(missing_cols)}")
groups_df["image_name"] = groups_df["image_name"].str.strip()
groups_df = groups_df[groups_df["image_name"].isin(set(disk_df["image_name"]))]
if groups_df["image_name"].duplicated().any():
    raise ValueError("groups.csv has duplicated rows for images in the zip: "
                     f"{groups_df.loc[groups_df['image_name'].duplicated(), 'image_name'].tolist()[:10]}")

dataset_df = disk_df.merge(groups_df[["image_name", "label", "group"]], on="image_name", how="left")
no_row = dataset_df[dataset_df["group"].isna() | (dataset_df["group"].str.strip() == "")]
if not no_row.empty:
    raise ValueError(f"{len(no_row)} images have no group in groups.csv, e.g. {no_row['image_name'].tolist()[:10]}")
mismatch = dataset_df[dataset_df["label"].str.strip() != dataset_df["folder"]]
if not mismatch.empty:
    raise ValueError(f"{len(mismatch)} images have a groups.csv label different from their folder, "
                     f"e.g. {mismatch[['image_name', 'label', 'folder']].head().to_dict('records')}")
dataset_df["group"] = dataset_df["group"].str.strip()
dataset_df = dataset_df.drop(columns="label")

print(f"Images: {len(dataset_df)} | groups: {dataset_df['group'].nunique()}")
print(dataset_df.groupby("class_name").agg(images=("image_name", "size"), groups=("group", "nunique")).to_string())

In [ ]:
%cd {HOME}

## Dataset split (group-aware, no leakage)

Frames of the same ad creative or the same source video are near-duplicates, so a random per-image split would leak them between train and evaluation. We therefore assign **whole groups** to **train/val/test** (target 70/15/15 of the frames of each class, fixed seed 42): no group appears in more than one split. Shares are approximate because groups have different sizes. The positive class is `junk-food-ad` (from the `ad/` folder) and the negative class is `non-junk-food-ad` (from the `non_ad/` folder).

Ultralytics expects an image-folder layout (`split/class/image.jpg`, with the validation folder named `val`), so the images are copied into `classification_dataset/{train,val,test}/{junk-food-ad,non-junk-food-ad}`. Ultralytics sorts class folders alphabetically, so `junk-food-ad` is class 0.

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
# When a class has fewer groups than splits, its groups go to the splits in this order:
# 1 group -> train; 2 groups -> train + test (so the class is still evaluated); >= 3 -> all splits.
SUPPORT_ORDER = ("train", "test", "val")


def stratified_group_split(frame, label_col, group_col="group", fractions=SPLIT_FRACTIONS,
                           support_order=SUPPORT_ORDER, seed=SEED):
    """Assign whole groups to splits while keeping the per-class frame shares close to `fractions`.

    Groups (same creative / same source video) are atomic, so near-identical frames never
    cross splits. A group may contain frames of several classes; it is described by its
    per-class frame counts. Classes are processed from the one with the fewest groups to
    the one with the most; each carrier group (largest first, seeded order on ties) goes to
    the split with the largest remaining deficit for that class, then for total frames.
    A repair pass then guarantees every class has at least one group in the splits given by
    `support_order` (moving the smallest donor group that its own split can spare).

    Returns a dict group -> split name.
    """
    splits = list(fractions)
    counts = pd.crosstab(frame[group_col], frame[label_col])
    order = counts.index.to_numpy().copy()
    np.random.default_rng(seed).shuffle(order)
    counts = counts.loc[order]
    classes = list(counts.columns)
    sizes = counts.sum(axis=1)

    class_target = {s: counts.sum(axis=0) * f for s, f in fractions.items()}
    frame_target = {s: sizes.sum() * f for s, f in fractions.items()}
    filled = {s: pd.Series(0.0, index=classes) for s in splits}
    assignment = {}

    n_carriers = (counts > 0).sum(axis=0).sort_values(kind="stable")
    for cls in n_carriers.index:
        carriers = [g for g in counts.index if g not in assignment and counts.loc[g, cls] > 0]
        carriers.sort(key=lambda g: -counts.loc[g, cls])
        for g in carriers:
            split = max(splits, key=lambda s: (class_target[s][cls] - filled[s][cls],
                                               frame_target[s] - filled[s].sum()))
            assignment[g] = split
            filled[split] += counts.loc[g]

    def required(cls):
        return [s for s in support_order if s in splits][:min(int(n_carriers[cls]), len(splits))]

    def carriers_in(cls, split):
        return sum(1 for g, s in assignment.items() if s == split and counts.loc[g, cls] > 0)

    def can_leave(g, split):
        return all(carriers_in(c, split) > 1 or split not in required(c)
                   for c in classes if counts.loc[g, c] > 0)

    changed = True
    while changed:
        changed = False
        for cls in n_carriers.index:
            for split in required(cls):
                if carriers_in(cls, split):
                    continue
                donors = [g for g, s in assignment.items()
                          if s != split and counts.loc[g, cls] > 0 and can_leave(g, s)]
                if donors:
                    assignment[min(donors, key=lambda g: sizes[g])] = split
                    changed = True

    for cls in n_carriers.index:
        missing = [s for s in required(cls) if not carriers_in(cls, s)]
        if missing:
            print(f"  NOTE: class {cls!r} has no group in {missing} (its groups cannot be moved)")
        if n_carriers[cls] < len(splits):
            present = [s for s in splits if carriers_in(cls, s)]
            print(f"  NOTE: class {cls!r} has only {int(n_carriers[cls])} group(s); present in {present}")
    return assignment


def assert_no_group_leakage(frame, group_col="group", split_col="split"):
    splits_per_group = frame.groupby(group_col)[split_col].nunique()
    leaked = splits_per_group[splits_per_group > 1]
    assert leaked.empty, f"LEAKAGE: groups in more than one split: {list(leaked.index)}"


def print_split_summary(frame, label_col, group_col="group", split_col="split"):
    split_names = [s for s in SPLIT_FRACTIONS if s in set(frame[split_col])]
    frames = pd.crosstab(frame[label_col], frame[split_col]).reindex(columns=split_names, fill_value=0)
    groups = (frame.groupby([label_col, split_col])[group_col].nunique().unstack(fill_value=0)
              .reindex(index=frames.index, columns=split_names, fill_value=0))
    table = frames.astype(str) + " (" + groups.astype(str) + " gr)"
    total_groups = frame.groupby(label_col)[group_col].nunique().reindex(frames.index)
    table["total"] = frames.sum(axis=1).astype(str) + " (" + total_groups.astype(str) + " gr)"
    print("Frames (groups) per class and split:")
    print(table.to_string())
    totals = frames.sum(axis=0)
    print("\nFrames per split: " + ", ".join(
        f"{s}={int(totals[s])} ({totals[s] / totals.sum():.1%}, "
        f"{frame.loc[frame[split_col] == s, group_col].nunique()} groups)"
        for s in split_names))


group_to_split = stratified_group_split(dataset_df, label_col="class_name")
dataset_df["split"] = dataset_df["group"].map(group_to_split)
assert dataset_df["split"].notna().all()
assert_no_group_leakage(dataset_df)
# Interleave classes inside every split (reproducible)
dataset_df = dataset_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print_split_summary(dataset_df, label_col="class_name")

CLASSIFICATION_DATASET_PATH = os.path.join(HOME, "classification_dataset")

print(f"\nBuilding classification dataset from: {DATASET_ROOT}")
print(f"Output will be saved to: {CLASSIFICATION_DATASET_PATH}\n")

# Start clean to avoid mixing with a previous run
if os.path.exists(CLASSIFICATION_DATASET_PATH):
    shutil.rmtree(CLASSIFICATION_DATASET_PATH)

for (split_name, class_name), part_df in dataset_df.groupby(["split", "class_name"]):
    out_dir = os.path.join(CLASSIFICATION_DATASET_PATH, split_name, class_name)
    os.makedirs(out_dir, exist_ok=True)
    for src_path, file_name in zip(part_df["path"], part_df["image_name"]):
        shutil.copy2(src_path, os.path.join(out_dir, file_name))
    print(f"  {split_name} - {class_name}: {len(part_df)} images")

os.makedirs(os.path.join(HOME, "runs"), exist_ok=True)
SPLIT_MANIFEST_PATH = os.path.join(HOME, "runs", "yolo_binary_split_manifest.csv")
dataset_df[["image_name", "folder", "class_name", "group", "split"]].to_csv(SPLIT_MANIFEST_PATH, index=False)
print("\nSplit manifest:", SPLIT_MANIFEST_PATH)
print("Done.")

## Train Classification YOLO11 model with dataset

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m-cls.pt')

model.train(
    data=CLASSIFICATION_DATASET_PATH,
    epochs=200,
    imgsz=640,
    device=0,
    patience=30
)

BEST_MODEL_PATH = os.path.join(model.trainer.save_dir, "weights", "best.pt")
print("Best model:", BEST_MODEL_PATH)

## Run predictions on test set

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import os
from IPython.display import Image, display

model = YOLO(BEST_MODEL_PATH)

results = model.val(
    data=CLASSIFICATION_DATASET_PATH,
    split='test',
    imgsz=640,
    device=0
)

print("\n\n" + "="*50)
print("TEST SET METRICS")
print("="*50)

if hasattr(results, 'confusion_matrix'):
    cm = results.confusion_matrix.matrix.astype(int)

    # Reconstruct y_true and y_pred from confusion matrix
    y_true = []
    y_pred = []
    for true_class in range(cm.shape[0]):
        for pred_class in range(cm.shape[1]):
            count = int(cm[pred_class, true_class])
            y_true.extend([true_class] * count)
            y_pred.extend([pred_class] * count)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    class_names = list(model.names.values()) if hasattr(model, 'names') else ['Class 0', 'Class 1']

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=0)
    recall = recall_score(y_true, y_pred, pos_label=0)
    f1 = f1_score(y_true, y_pred, pos_label=0)

    print(f"Accuracy: {accuracy:.3f}")
    print(f"Accuracy from YOLO results: {results.top1}")
    print(f"Precision: {precision:.3f} ({class_names[0]})")
    print(f"Recall: {recall:.3f} ({class_names[0]})")
    print(f"F1-Score: {f1:.3f} ({class_names[0]})")

cm_path = os.path.join(results.save_dir, 'confusion_matrix.png')
if os.path.exists(cm_path):
    print(f"Confusion Matrix:")
    display(Image(filename=cm_path))

## Real images test

Let's test the model on random images from the test set to see the prediction with a visual display.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

# Get test images from the junk-food-ad folder only (positive class)
# We only want to display interesting images that have actual junk food ads
test_dir = os.path.join(CLASSIFICATION_DATASET_PATH, "test")
positive_class_dir = os.path.join(test_dir, "junk-food-ad")
test_images = []
for img_file in os.listdir(positive_class_dir):
    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
        test_images.append(os.path.join(positive_class_dir, img_file))

# Pick 5 random test images from positive class
random_images = random.sample(test_images, min(5, len(test_images)))

# Load the best model
best_model = YOLO(BEST_MODEL_PATH)

# Define colors for labels
colors = {
    "junk-food-ad": "#FF6B6B",      # Red
    "non-junk-food-ad": "#4ECDC4"   # Teal
}
default_color = "#95A5A6"  # Gray for unknown classes

for idx, random_image_path in enumerate(random_images, 1):
    print(f"{'='*60}")
    print(f"Image {idx}/5: {random_image_path}")
    print('='*60)

    # Run inference
    result = best_model(random_image_path, verbose=False)[0]
    probs = result.probs
    predicted_class_idx = probs.top1
    predicted_class_name = best_model.names[predicted_class_idx]
    confidence = probs.top1conf.item()

    # Get true label from folder name
    true_label = os.path.basename(os.path.dirname(random_image_path))
    is_correct = predicted_class_name == true_label

    # Load and display the image
    img = PILImage.open(random_image_path).convert("RGB")
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img)
    ax.axis("off")

    # Create label badge
    label_color = colors.get(predicted_class_name, default_color)
    label_text = predicted_class_name.upper()

    # Add colored badge at the bottom
    badge = mpatches.FancyBboxPatch(
        (0.25, -0.08), 0.5, 0.06,
        boxstyle="round,pad=0.01",
        facecolor=label_color,
        edgecolor="white",
        linewidth=2,
        transform=ax.transAxes,
        clip_on=False
    )
    ax.add_patch(badge)
    ax.text(
        0.5, -0.05,
        label_text,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        color="white",
        ha="center",
        va="center"
    )

    plt.tight_layout()
    plt.show()

    print(f"Prediction: {predicted_class_name} (confidence: {confidence:.2%})")
    print(f"True label: {true_label}")
    print(f"Correct: {is_correct}")
